# 5G Network Intelligence — Probability Calibration Study

## Hypothesis

Can an affine-logit correction improve active-load probability and the mixture median? Improved log loss need not improve MAE or zero detection. Preserve all negative results.

## Temporal protocol

Reuse frozen development classifiers and positive quantile models. Fit one fixed-L2 logistic correction on each fold's stopping labels; evaluate only on its later scoring labels. Stopping data also selected the base model rounds: this limitation is explicit. No calibration on scoring targets, reused validation 504–671 or final holdout. These development folds supported prior research and cannot establish fresh-test generalization.


In [ ]:
RUN_CALIBRATION = False
SOURCE_COMMIT = 'aa748527eaf64a20704b01ac33551e4f6151070b'


## Fixed configuration

One method, C=1, no hyperparameter search. Six calibrators fit in hosted Kaggle only. The base trees and quantile models are unchanged.


In [ ]:
CALIBRATION_PROTOCOL = {'protocol': 'hurdle-affine-logit-development-v1',
 'method': 'fixed_L2_logistic_regression_on_classifier_logit',
 'C': 1.0,
 'solver': 'lbfgs',
 'max_iter': 200,
 'tol': 1e-06,
 'epsilon': 1e-06,
 'seed': 17,
 'base_checkpoints': 'artifacts/hurdle_research',
 'calibration_population': 'each_development_fold_stopping_targets_only',
 'scoring_population': 'each_development_fold_scoring_targets_only',
 'base_model_retraining': False,
 'quantile_retraining': False,
 'selection_on_validation_504_671': 'forbidden',
 'original_final_holdout': 'forbidden',
 'production_change': 'forbidden',
 'limitation': 'calibration_window_also_selected_base_component_rounds_so_scoring_window_must_remain_separate'}


## Probability transformation

Clip probabilities only for numerical stability, then apply sigmoid(a × logit(p) + b). The following cell defines inference arithmetic only. The two coefficients are fitted on cloud in a later guarded cell.


In [ ]:
"""Pure inference for a saved affine-logit probability calibrator."""

import numpy as np


def probability_logits(probability, epsilon=1e-6):
    probability = np.asarray(probability, dtype=float)
    if (not 0 < epsilon < 0.5 or not np.isfinite(probability).all()
            or ((probability < 0) | (probability > 1)).any()):
        raise ValueError("Finite probabilities in [0,1] and valid clipping epsilon required")
    clipped = np.clip(probability, epsilon, 1 - epsilon)
    return np.log(clipped) - np.log1p(-clipped)


def calibrated_probability(probability, slope, intercept, epsilon=1e-6):
    if not np.isfinite([slope, intercept]).all():
        raise ValueError("Finite scalar calibration coefficients required")
    logits = slope * probability_logits(probability, epsilon) + intercept
    return np.exp(-np.logaddexp(0, -logits))


## Hosted environment and saved checkpoints

Attach the completed original zero-inflated development output and the MIT calibration source dataset. No repository source dump is embedded in this notebook. Base checkpoint hashes are checked before calibration.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, tarfile
if RUN_CALIBRATION:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Calibration requires hosted Kaggle')
    roots = [p.parent.parent for p in Path('/kaggle/input').rglob('train_hurdle_research.py') if (p.parent.parent/'artifacts/hurdle_research/manifest.json').exists()]
    if len(roots) != 1:
        raise RuntimeError('Attach exactly one development output')
    PROJECT = Path('/kaggle/working/hurdle_calibration')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('src','scripts','configs','data/processed','artifacts/hurdle_research'):
        shutil.copytree(roots[0]/folder, PROJECT/folder, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0,str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT'] = SOURCE_COMMIT
    Path('reports').mkdir(exist_ok=True)


## Install the focused implementation

Only the calibration trainer, inference helper and declared configuration come from the source attachment. Fitting code remains in `scripts/calibrate_hurdle_development.py` for focused review and debugging.


In [ ]:
if RUN_CALIBRATION:
    patches = list(Path('/kaggle/input').rglob('calibrate_hurdle_development.py'))
    if len(patches) != 1:
        raise RuntimeError('Attach exactly one calibration source')
    patch = patches[0].parent
    for name, target in (('calibrate_hurdle_development.py','scripts'),('probability_calibration.py','src/models'),('hurdle_calibration.yaml','configs')):
        shutil.copyfile(patch/name, Path(target)/name)


## Cloud calibration and debugging

Convergence failures stop the experiment. Save coefficients, scoring probabilities and predictions for independent verification. No new base-model fitting or quantile selection occurs.


In [ ]:
if RUN_CALIBRATION:
    subprocess.run([sys.executable,'-m','pip','install','lightgbm>=4.6,<5','scikit-learn>=1.7,<1.10','numpy','pandas','pyyaml'], check=True)
    with Path('reports/hurdle_calibration.log').open('w') as log:
        process = subprocess.Popen([sys.executable,'-u','-m','scripts.calibrate_hurdle_development'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        status = process.wait()
    with tarfile.open('/kaggle/working/hurdle_calibration_results.tar.gz','w:gz') as archive:
        for folder in ('reports','artifacts/hurdle_calibration'):
            if Path(folder).exists():
                archive.add(folder,arcname=folder)
    if status:
        raise RuntimeError('Calibration failed; inspect saved log')
else:
    print('Review mode: no fitting or calibration.')


## Saved scoring results and limitations

MAE uses original units; WAPE is a ratio. Report Brier score, log loss and zero false-positive rate with every fold/horizon. Compare with raw probabilities on exactly the same scoring targets. A calibration claim requires evidence, and these exploratory scores cannot establish no overfitting. [Method reference](https://scikit-learn.org/stable/modules/calibration.html).


In [ ]:
paths = [Path('reports/hurdle_calibration_development.csv'),Path('../reports/hurdle_calibration_development.csv')]
if Path('/kaggle/input').is_dir():
    paths += list(Path('/kaggle/input').rglob('hurdle_calibration_development.csv'))
path = next((p for p in paths if p.exists()),None)
if path:
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(path))
else:
    print('No completed calibration scores attached.')
